In [1]:
# pinn_soh_forecast
# Physics-informed NN for SOH forecasting (daily timestep)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from datetime import timedelta

In [2]:
# -----------------------
# User parameters
# -----------------------
csv_path = "battery_timeseries.csv"
resample_rule = '1D' # aggregate to daily resolution
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 128
epochs = 100
lr = 1e-3
R_const = 8.314  # J/mol/K
# You can set Ea constants or leave them trainable; here we set initial guesses
Ea_storage_init = 25000.0
Ea_cycling_init = 35000.0
beta_init = 0.5  # exponent for throughput dependence

# Loss weighting between data fit and physics consistency (tune)
lambda_physics = 1.0

In [ ]:
# -----------------------
# Example run
# -----------------------
# if __name__ == "__main__":
model, scaler_X, scaler_y, df_enriched, features = train_pipeline(csv_path)
# take last available day as starting point
last_row = df_enriched.iloc[-1]
pred_soh = predict_future_soh(model, scaler_X, scaler_y, last_row, days_to_predict=365*5)
# plot
plt.figure(figsize=(8,4))
days = np.arange(len(pred_soh))
plt.plot(days, pred_soh, label='Predicted SOH')
plt.xlabel('Days from now')
plt.ylabel('SOH (0..1)')
plt.title('SOH projection (5 years)')
plt.grid(True)
plt.show()

In [3]:
# -----------------------
# Helpers: load & preprocess
# -----------------------
def load_and_aggregate(csv_path, resample_rule='1D'):
    df = pd.read_csv(csv_path, parse_dates=['time'])
    df = df.sort_values('time').set_index('time')
    # Resample to daily aggregates: mean for continuous, sum for throughput
    agg = {}
    # features to average
    avg_cols = ['batt_voltage', 'soc', 'soh'] + \
               [f'temperature_M{i}' for i in range(1,6)] + \
               [f'cell_voltages_{i}' for i in range(1,14)]
    for c in avg_cols:
        if c in df.columns:
            agg[c] = 'mean'
    # current: we need throughput = sum(abs(I)*dt_hours)
    # approximate by mean(|I|)*24 if resampled daily and data dense; else compute exact below
    # for generality compute daily Ah throughput by integrating |I|*dt
    # so we will compute throughput column separately
    df['abs_current'] = df['batt_current'].abs()
    # compute exact Ah per sample if time spacing irregular
    df = df.copy()
    # approximate area under |I| over each interval: I(t) * dt_hours
    df['dt_hours'] = df.index.to_series().diff().dt.total_seconds().div(3600).fillna(0)
    df['Ah_integral'] = (df['abs_current'] * df['dt_hours']).fillna(0)  # Ah
    # Now resample
    daily = df.resample(resample_rule).agg(agg)
    daily['Ah_per_day'] = df['Ah_integral'].resample(resample_rule).sum().values
    # also compute cycle proxy: cumulative Ah / (2*Q_nom) could approximate full cycles if Q_nom known
    daily = daily.dropna(subset=['soh'])  # we need soh for supervised training
    daily = daily.reset_index()
    return daily

In [4]:
# -----------------------
# Feature engineering
# -----------------------
def make_features(df_daily, Q_nom=116.0):
    """
    Input: daily aggregated df with columns created above
    Output: features X and target y (delta_soh per day)
    """
    df = df_daily.copy()
    # compute daily dSoH (forward difference)
    df['soh_next'] = df['soh'].shift(-1)
    df = df[:-1]  # drop last since no next
    df['delta_soh'] = df['soh_next'] - df['soh']  # negative expected
    # throughputs
    df['Ah_day'] = df['Ah_per_day']
    df['cum_Ah'] = df['Ah_day'].cumsum()
    # cell voltage stats
    cell_cols = [f'cell_voltages_{i}' for i in range(1,14)]
    df['cell_v_mean'] = df[cell_cols].mean(axis=1)
    df['cell_v_std'] = df[cell_cols].std(axis=1)
    df['temp_mean'] = df[[f'temperature_M{i}' for i in range(1,6)]].mean(axis=1)
    # simple cycle proxy (approx cycles per day) = Ah_day / Q_nom
    df['cycles_day_proxy'] = df['Ah_day'] / Q_nom
    # SOC stats
    df['soc_mean'] = df['soc']
    # features list
    features = ['soh', 'soc_mean', 'batt_voltage', 'Ah_day', 'cum_Ah',
                'cycles_day_proxy', 'temp_mean', 'cell_v_mean', 'cell_v_std']
    # Drop any missing
    df = df.dropna(subset=features + ['delta_soh'])
    X = df[features].values
    y = df['delta_soh'].values.reshape(-1,1)
    dates = df['time'].values
    return X, y, df, features, dates

In [5]:
# -----------------------
# Physics model components (vectorized)
# -----------------------
def physics_storage_rate(SOC, T_K, k_s_factor, Ea_s, alpha, t_day):
    # returns predicted delta_soh/day contribution from storage-like aging
    # k_s_factor (learned) multiplies this
    # we implement: Q_loss_storage_per_day = k_s * f(SOC) * exp(-Ea/(R*T)) * t^{alpha-1} approximated
    # For simplicity treat t_day ~ 1 day, and produce rate proportional to exp(-Ea/(R*T)) * f(SOC)
    arr = k_s_factor * torch.exp(-Ea_s / (R_const * T_K)) * f_soc_nn(SOC)
    # arr shape batch x 1
    return arr

def physics_cycling_rate(SOC_max, SOC_min, Crate_proxy, T_K, k_c_factor, Ea_c, beta, throughput_day):
    # Cycling contribution ~ k_c * f(soc_max)*f(soc_min)*f(Crate)*exp(-Ea/(R*T)) * throughput^beta
    arr = k_c_factor * torch.exp(-Ea_c / (R_const * T_K)) \
          * f_socmax_nn(SOC_max) * f_socmin_nn(SOC_min) * f_crate_nn(Crate_proxy) \
          * (throughput_day ** beta)
    return arr

In [6]:
# -----------------------
# Define NN modules
# -----------------------
class SmallPositiveNet(nn.Module):
    """ Small net that outputs a positive scalar factor (using softplus). """
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8,1),
        )
    def forward(self, x):
        return F.softplus(self.net(x)) + 1e-6

In [7]:
class ResidualNet(nn.Module):
    """ Residual network that learns remaining effects """
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.net(x)

In [8]:
class PINN_SOHR(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # small nets for multiplicative f(SOC), f(SOC_max), f(SOC_min), f(Crate)
        self.f_soc = SmallPositiveNet(1)
        self.f_socmax = SmallPositiveNet(1)
        self.f_socmin = SmallPositiveNet(1)
        self.f_crate = SmallPositiveNet(1)
        # small nets that output k_s and k_c (positive)
        self.k_s_net = SmallPositiveNet(n_features)
        self.k_c_net = SmallPositiveNet(n_features)
        # optionally make Ea and exponents trainable scalars
        self.log_Ea_s = nn.Parameter(torch.log(torch.tensor(Ea_storage_init)))
        self.log_Ea_c = nn.Parameter(torch.log(torch.tensor(Ea_cycling_init)))
        self.log_beta = nn.Parameter(torch.log(torch.tensor(beta_init)))
        # residual net
        self.res_net = ResidualNet(n_features)
    def forward(self, X_batch_tensor, T_K_tensor, throughput_day_tensor, soc_tensor, socmax_tensor, socmin_tensor, crate_tensor):
        """
        X_batch_tensor not directly used but kept if needed
        Inputs are torch tensors shaped [batch, ...]
        """
        # compute physics multiplicative factors
        k_s = self.k_s_net(X_batch_tensor)  # [B,1]
        k_c = self.k_c_net(X_batch_tensor)  # [B,1]
        Ea_s = torch.exp(self.log_Ea_s)
        Ea_c = torch.exp(self.log_Ea_c)
        beta = torch.exp(self.log_beta)

        f_soc_val = self.f_soc(soc_tensor)
        f_socmax_val = self.f_socmax(socmax_tensor)
        f_socmin_val = self.f_socmin(socmin_tensor)
        f_crate_val = self.f_crate(crate_tensor)

        # physics storage part
        phys_storage = k_s * torch.exp(-Ea_s / (R_const * T_K_tensor)) * f_soc_val
        # physics cycling part (throughput_day_tensor ** beta)
        phys_cycling = k_c * torch.exp(-Ea_c / (R_const * T_K_tensor)) \
                       * f_socmax_val * f_socmin_val * f_crate_val * (throughput_day_tensor ** beta)

        phys_total = phys_storage + phys_cycling  # predicted delta_soh/day by physics component

        # residual correction
        residual = self.res_net(X_batch_tensor)  # can be positive or negative

        # total predicted delta SoH per day
        delta_pred = phys_total + residual
        return delta_pred, phys_total, residual

In [9]:
# We'll map the small f*_nn calls to these instances (set below)
f_soc_nn = None
f_socmax_nn = None
f_socmin_nn = None
f_crate_nn = None

# -----------------------
# Training pipeline
# -----------------------
def train_pipeline(csv_path):
    # load data
    daily = load_and_aggregate(csv_path, resample_rule=resample_rule)
    X_raw, y_raw, df_enriched, features, dates = make_features(daily)
    print("Feature names:", features)
    # scale features
    scaler_X = StandardScaler().fit(X_raw)
    scaler_y = StandardScaler().fit(y_raw)
    X_scaled = scaler_X.transform(X_raw)
    y_scaled = scaler_y.transform(y_raw)

    # Prepare tensors
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32).to(device)

    # Build additional tensors for physics inputs (unscaled)
    # Map columns indices
    idx_soh = features.index('soh')
    idx_soc = features.index('soc_mean')
    idx_voltage = features.index('batt_voltage')
    idx_ahday = features.index('Ah_day')
    idx_cumah = features.index('cum_Ah')
    idx_cycles_proxy = features.index('cycles_day_proxy')
    idx_temp = features.index('temp_mean')

    soc_t = torch.tensor(X_raw[:, idx_soc:idx_soc+1], dtype=torch.float32).to(device)
    # for socmax & min approximations, use soc +/- small windows or use soc itself as proxy
    # here we use soc as both max and min placeholders (user should provide better signals if available)
    socmax_t = soc_t.clone()
    socmin_t = soc_t.clone()
    temp_K_t = torch.tensor(X_raw[:, idx_temp:idx_temp+1] + 273.15, dtype=torch.float32).to(device)
    throughput_day_t = torch.tensor(X_raw[:, idx_ahday:idx_ahday+1], dtype=torch.float32).to(device)
    crate_t = torch.tensor(X_raw[:, idx_cycles_proxy:idx_cycles_proxy+1], dtype=torch.float32).to(device)

    # Build model
    global f_soc_nn, f_socmax_nn, f_socmin_nn, f_crate_nn
    model = PINN_SOHR(n_features=X_scaled.shape[1]).to(device)
    # bind small nets for physics functions
    f_soc_nn = model.f_soc
    f_socmax_nn = model.f_socmax
    f_socmin_nn = model.f_socmin
    f_crate_nn = model.f_crate

    # DataLoader
    dataset = TensorDataset(X_tensor, y_tensor, soc_t, socmax_t, socmin_t, temp_K_t, throughput_day_t, crate_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    # training loop
    for ep in range(epochs):
        model.train()
        epoch_loss = 0.0
        for batch in loader:
            Xb, yb, socb, socmaxb, socminb, tempKb, throughputb, crateb = batch
            opt.zero_grad()
            delta_pred_scaled, phys_pred_scaled, resid_scaled = model(Xb, tempKb, throughputb, socb, socmaxb, socminb, crateb)
            # delta_pred_scaled has shape [B,1] in original scale (we built physics on raw)
            # Our y_tensor was scaled; so transform delta_pred to scaled space before MSE:
            # easiest is to inverse-scale yb and compute loss in raw space:
            # but yb is scaled, so compute yb_inv = scaler_y.inverse_transform(yb.cpu().numpy())
            # Simpler: compute loss in raw space by un-scaling yb and compare to delta_pred (raw)
            yb_raw = torch.tensor(scaler_y.inverse_transform(yb.cpu().numpy()), dtype=torch.float32).to(device)
            # physics consistency loss: encourage phys_pred to be near delta_pred (optional)
            loss_data = mse(delta_pred_scaled, yb_raw)
            # Optionally penalize extremely negative rates or non-physical behavior by L2 on phys vs data
            loss_phys = mse(phys_pred_scaled, yb_raw)
            loss = loss_data + lambda_physics * loss_phys
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * Xb.shape[0]
        epoch_loss /= len(dataset)
        if ep % 5 == 0 or ep == epochs-1:
            print(f"Epoch {ep:03d} Loss={epoch_loss:.6e}")
    # Save model and scalers
    torch.save(model.state_dict(), 'pinn_soh_model.pth')
    import joblib
    joblib.dump(scaler_X, 'scaler_X.pkl')
    joblib.dump(scaler_y, 'scaler_y.pkl')
    print("Training finished.")
    return model, scaler_X, scaler_y, df_enriched, features

In [10]:
# -----------------------
# Prediction forward for N days
# -----------------------
def predict_future_soh(model, scaler_X, scaler_y, recent_df_row, days_to_predict=365*5):
    """
    recent_df_row: DataFrame row (latest available day) with same columns used in features
    Option A: you can pass a user-supplied future profile (temp, Ah_day, soc) as a DataFrame;
    Here we repeat the last day's conditions for simplicity.
    """
    model.eval()
    # build starting point
    current_soh = float(recent_df_row['soh'])
    # use last row features as baseline
    fea = recent_df_row.copy()
    # create future rows repeating last row (or you can accept a DataFrame of future profiles)
    future = []
    for d in range(days_to_predict):
        future.append(fea.copy())
    future_df = pd.DataFrame(future).reset_index(drop=True)
    # now iterate day by day
    soh_series = [current_soh]
    for i in range(days_to_predict):
        row = future_df.iloc[i]
        X_row = np.array([row[features].values])
        Xs = scaler_X.transform(X_row)
        Xb = torch.tensor(Xs, dtype=torch.float32).to(device)
        # build physics tensors
        socb = torch.tensor([[row['soc_mean']]], dtype=torch.float32).to(device)
        socmaxb = socb.clone()
        socminb = socb.clone()
        tempKb = torch.tensor([[row['temp_mean'] + 273.15]], dtype=torch.float32).to(device)
        throughputb = torch.tensor([[row['Ah_day']]], dtype=torch.float32).to(device)
        crateb = torch.tensor([[row['cycles_day_proxy']]], dtype=torch.float32).to(device)
        with torch.no_grad():
            delta_pred, phys, resid = model(Xb, tempKb, throughputb, socb, socmaxb, socminb, crateb)
        # delta_pred is predicted delta SoH per day (raw)
        # update soh
        current_soh = float(np.clip(current_soh + float(delta_pred.cpu().numpy().flatten()[0]), 0.0, 1.0))
        soh_series.append(current_soh)
    # return vector of predicted soh (length days_to_predict+1)
    return soh_series

FileNotFoundError: [Errno 2] No such file or directory: 'battery_timeseries.csv'